# Tutorial 2-1: Data Loading, Visualization, and Cleaning
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

## Objective
In this tutorial, we move beyond the classroom slides to handle data as it exists in the real world: messy, incomplete, and noisy. We will explore:
1. **The Anatomy of Data:** Understanding how `pandas` represents objects (records) and attributes (features).
2. **Data Inspection:** Visualizing missing patterns to decide on a cleaning strategy.
3. **Cleaning Missing Values:** Comparing deletion versus imputation (filling in the gaps).
4. **Outlier Detection:** Using statistical methods (IQR) to find 'noisy' data points.
5. **Duplicate Management:** Ensuring our dataset doesn't contain redundant information.

## 1. Setup and Data Loading
We will use the **Titanic dataset** provided by the `seaborn` library. It is a perfect 'messy' dataset because it contains missing ages, missing locations, and categorical variables that need attention.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = sns.load_dataset('titanic')

# Display the 'Anatomy' of the data
print(f"Dataset Shape: {df.shape} (Objects x Attributes)")
df.head()

## 2. Inspecting Data Quality
Before cleaning, we must diagnose the problem. We use `.info()` to check data types (Nominal vs. Ratio) and `.isnull()` to find missing values.

In [ ]:
# Summary of attributes and their types
print(df.info())

# Visualizing missing data
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title("Missing Data Heatmap (Yellow lines indicate missing values)")
plt.show()

## 3. Handling Missing Values
In class, we discussed two main strategies: **Eliminate** or **Estimate**.

### 3.1 Elimination (Dropping)
If an attribute is missing too much data (like 'deck' above), it may be better to remove the entire column. If a record is missing the target value, we remove the row.

In [ ]:
# Drop 'deck' column because it's mostly empty
df_cleaned = df.drop(columns=['deck'])

# Drop rows where 'embarked' is missing (only a few rows)
df_cleaned.dropna(subset=['embarked'], inplace=True)

print(f"New shape after dropping: {df_cleaned.shape}")

### 3.2 Estimation (Imputation)
For the 'age' attribute, we don't want to throw away 20% of our data. Instead, we fill in the missing values. 
**Theory Check:** Why use the *median* instead of the *mean*? The median is robust to outliers!

In [ ]:
# Calculate median age
median_age = df_cleaned['age'].median()
print(f"Imputing missing ages with Median: {median_age}")

# Fill gaps
df_cleaned['age'] = df_cleaned['age'].fillna(median_age)

# Verify no more missing values
print("Remaining missing values:")
print(df_cleaned[['age', 'embarked']].isnull().sum())

## 4. Detecting Outliers (Noise)
Outliers are data points that differ significantly from other observations. We can detect them using the **Interquartile Range (IQR) method**.

**The Logic:** Any value below $Q1 - 1.5 \times IQR$ or above $Q3 + 1.5 \times IQR$ is considered an outlier.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=df_cleaned['fare'])
plt.title("Detecting Outliers in 'Fare' Attribute")
plt.show()

# Calculate IQR
Q1 = df_cleaned['fare'].quantile(0.25)
Q3 = df_cleaned['fare'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_cleaned[(df_cleaned['fare'] < lower_bound) | (df_cleaned['fare'] > upper_bound)]
print(f"Number of outliers detected in 'Fare': {len(outliers)}")

## 5. Identifying Duplicate Data
Duplicates often occur when merging heterogeneous data sources. We need to identify and remove them to prevent our model from being biased toward redundant samples.

In [ ]:
# Count duplicates
duplicate_count = df_cleaned.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

# Remove duplicates
df_final = df_cleaned.drop_duplicates()
print(f"Final Dataset Shape: {df_final.shape}")

## Summary
We have successfully taken a raw, 'dirty' dataset and:
1. Identified the missing value patterns visually.
2. Decided when to drop data (high missing count) and when to impute (low missing count).
3. Used the IQR method to flag potential noise in our numerical attributes.
4. Cleaned duplicates to ensure data integrity.

In the next tutorial, we will transform these cleaned attributes into formats more suitable for machine learning algorithms!